# PixelPlay word-sync model: singing fine-tune on real songs (free Kaggle GPU)
1. Right panel → **Session options**: Accelerator **GPU T4 x2**, Internet **On** (phone-verified account).
2. **Recommended: Add Input → your `kaggle_data_prep` notebook's output** (all datasets, already downloaded). Without it this notebook downloads everything itself, which can take many hours when Zenodo is slow.
3. Optional, best data: your own songs. Make a private Kaggle Dataset of audio files with a same-named synced `.lrc` next to each (e.g. `Song.mp3` + `Song.lrc`), then **Add Input** it here.
4. Optional: Add-ons → Secrets → `HF_TOKEN` to upload the finished model.
5. **Save Version → Save & Run All (Commit)**. Runs in the background; results in the **Output** tab.

In [ ]:
STEPS, LR = 6000, 3e-5
USE_GTSINGER = False                      # slow (Hugging Face rate limits) and gave 0 usable clips in run 1
BASE    = "facebook/wav2vec2-base-960h"   # or a previous run's best/ folder attached as input
VERSION = "w2v2-base-960h-singing-int8-v1"
HF_REPO = ""
DATA    = "/tmp/pp_data"
RUN     = "/kaggle/working/runs/singing-v1"
KIT     = "/kaggle/working/singing_finetune"
MY_MUSIC = "/kaggle/input"                # any attached dataset with .lrc files is used
PREPARED_SOURCES = ["/kaggle/input"]      # attach the output of kaggle_data_prep.ipynb to skip downloads
import os, subprocess, glob, shutil
for d in (DATA, RUN): os.makedirs(d, exist_ok=True)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv; df -h /tmp /kaggle/working", shell=True, capture_output=True, text=True).stdout)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN"); print("HF_TOKEN loaded")
except Exception:
    print("no HF_TOKEN secret (only needed for the upload)")
prev = glob.glob("/kaggle/input/**/runs/singing-v1/resume.pt", recursive=True)
if prev and not os.path.exists(f"{RUN}/resume.pt"):
    shutil.copytree(os.path.dirname(prev[0]), RUN, dirs_exist_ok=True); print("resuming from", prev[0])

In [ ]:
import base64, os
os.makedirs(KIT, exist_ok=True)
FILES = {
"common.py": "IiIiU2hhcmVkIGhlbHBlcnMgZm9yIHRoZSBzaW5naW5nIGZpbmUtdHVuZSBraXQuCgpFdmVyeXRoaW5nIGhlcmUgbWlycm9ycyB3aGF0IHRoZSBhcHAgZG9lcyBvbiB0aGUgcGhvbmUsIHNvIGEgbW9kZWwgdHJhaW5lZCBhbmQgbWVhc3VyZWQgd2l0aAp0aGVzZSBzY3JpcHRzIGJlaGF2ZXMgdGhlIHNhbWUgd2F5IG9uY2UgaXQgaXMgc3dhcHBlZCBpbnRvIGBBbGlnbm1lbnRNb2RlbE1hbmFnZXJgOgoKKiB0aGUgdm9jYWJ1bGFyeSBpcyBgZmFjZWJvb2svd2F2MnZlYzItYmFzZS05NjBoYCdzLCB0b2tlbiBmb3IgdG9rZW4gKGBDdGNWb2NhYnVsYXJ5LldBVjJWRUMyXzk2MEhgKTsKKiBhdWRpbyBpcyAxNiBrSHogbW9ubywgbm9ybWFsaXNlZCB0byB6ZXJvIG1lYW4gLyB1bml0IHZhcmlhbmNlIChgQ2h1bmtlZEVtaXNzaW9ucy5ub3JtYWxpemVgKTsKKiBvbmUgbW9kZWwgZnJhbWUgaXMgMzIwIHNhbXBsZXMgKDIwIG1zKS4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBqc29uCmltcG9ydCBtYXRoCmltcG9ydCByZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAoKU0FNUExFX1JBVEUgPSAxNl8wMDAKSE9QID0gMzIwICAjIHNhbXBsZXMgcGVyIG1vZGVsIGZyYW1lICgyMCBtcykKQkFTRV9NT0RFTCA9ICJmYWNlYm9vay93YXYydmVjMi1iYXNlLTk2MGgiCgpCTEFOSyA9IDAKV09SRF9TRVAgPSA0CkxFVFRFUlMgPSB7CiAgICAiRSI6IDUsICJUIjogNiwgIkEiOiA3LCAiTyI6IDgsICJOIjogOSwgIkkiOiAxMCwgIkgiOiAxMSwgIlMiOiAxMiwKICAgICJSIjogMTMsICJEIjogMTQsICJMIjogMTUsICJVIjogMTYsICJNIjogMTcsICJXIjogMTgsICJDIjogMTksICJGIjogMjAsCiAgICAiRyI6IDIxLCAiWSI6IDIyLCAiUCI6IDIzLCAiQiI6IDI0LCAiViI6IDI1LCAiSyI6IDI2LCAiJyI6IDI3LCAiWCI6IDI4LAogICAgIkoiOiAyOSwgIlEiOiAzMCwgIloiOiAzMSwKfQpWT0NBQl9TSVpFID0gMzIKSURfVE9fQ0hBUiA9IHt2OiBrIGZvciBrLCB2IGluIExFVFRFUlMuaXRlbXMoKX0KSURfVE9fQ0hBUltXT1JEX1NFUF0gPSAiICIKCiMgU2lsZW5jZSAvIGJyZWF0aCAvIGZpbGxlciBtYXJrZXJzIHVzZWQgYnkgc2luZ2luZyBjb3Jwb3JhIChHVFNpbmdlciwgT3BlbmNwb3Atc3R5bGUgbGFiZWxz4oCmKS4KX05PTl9MRVhJQ0FMID0geyJTUCIsICJBUCIsICI8U1A+IiwgIjxBUD4iLCAiU0lMIiwgIjxTSUw+IiwgIkJSIiwgIjxCUj4iLCAiUEFVIiwgIjxQQVU+IiwgIiMifQoKCmRlZiBub3JtYWxpemVfd29yZHModGV4dDogc3RyKSAtPiBsaXN0W3N0cl06CiAgICAiIiJEaXNwbGF5IGx5cmljcyAtPiBsaXN0IG9mIHVwcGVyLWNhc2Ugd29yZHMgY29udGFpbmluZyBvbmx5IG1vZGVsIGxldHRlcnMuCgogICAgU2FtZSBydWxlcyBhcyB0aGUgYXBwJ3MgYEFsaWdubWVudFRleHROb3JtYWxpemVyYDogcGFyZW50aGVzZXMgKGJhY2tpbmcgdm9jYWxzKSBhcmUgZHJvcHBlZCwKICAgIGN1cmx5IGFwb3N0cm9waGVzIGJlY29tZSBzdHJhaWdodCBvbmVzLCBldmVyeXRoaW5nIGVsc2UgdGhhdCBpcyBub3QgQS1aIG9yICcgc3BsaXRzIHdvcmRzLgogICAgV29yZHMgd2l0aCBkaWdpdHMgYXJlIHJldHVybmVkIGFzLWlzIHNvIGNhbGxlcnMgY2FuIHJlamVjdCB0aGUgdXR0ZXJhbmNlICh0aGUgbW9kZWwgd291bGQKICAgIHNwZWxsICIyNC83IiBkaWZmZXJlbnRseSBmcm9tIGhvdyBpdCBpcyBzdW5nKS4KICAgICIiIgogICAgdGV4dCA9IHJlLnN1YihyIlwoW14pXSpcKXxcW1teXF1dKlxdIiwgIiAiLCB0ZXh0KQogICAgdGV4dCA9IHRleHQucmVwbGFjZSgi4oCZIiwgIiciKS5yZXBsYWNlKCLigJgiLCAiJyIpLnJlcGxhY2UoImAiLCAiJyIpCiAgICBvdXQgPSBbXQogICAgZm9yIHJhdyBpbiB0ZXh0LnVwcGVyKCkuc3BsaXQoKToKICAgICAgICBpZiByYXcuc3RyaXAoIjw+IikgaW4ge3Quc3RyaXAoIjw+IikgZm9yIHQgaW4gX05PTl9MRVhJQ0FMfToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBhbnkoY2guaXNkaWdpdCgpIGZvciBjaCBpbiByYXcpOgogICAgICAgICAgICBvdXQuYXBwZW5kKHJhdykKICAgICAgICAgICAgY29udGludWUKICAgICAgICBmb3IgdyBpbiByZS5zcGxpdChyIlteQS1aJ10rIiwgcmF3KToKICAgICAgICAgICAgdyA9IHcuc3RyaXAoIiciKQogICAgICAgICAgICBpZiB3OgogICAgICAgICAgICAgICAgb3V0LmFwcGVuZCh3KQogICAgcmV0dXJuIG91dAoKCmRlZiB3b3Jkc190b19pZHMod29yZHM6IGxpc3Rbc3RyXSkgLT4gbGlzdFtpbnRdOgogICAgaWRzOiBsaXN0W2ludF0gPSBbXQogICAgZm9yIGksIHcgaW4gZW51bWVyYXRlKHdvcmRzKToKICAgICAgICBpZiBpOgogICAgICAgICAgICBpZHMuYXBwZW5kKFdPUkRfU0VQKQogICAgICAgIGlkcy5leHRlbmQoTEVUVEVSU1tjXSBmb3IgYyBpbiB3KQogICAgcmV0dXJuIGlkcwoKCmRlZiBoYXNfZGlnaXRzKHdvcmRzOiBsaXN0W3N0cl0pIC0+IGJvb2w6CiAgICByZXR1cm4gYW55KGFueShjaC5pc2RpZ2l0KCkgZm9yIGNoIGluIHcpIGZvciB3IGluIHdvcmRzKQoKCmRlZiBpZHNfdG9fdGV4dChpZHMpIC0+IHN0cjoKICAgIHJldHVybiAiIi5qb2luKElEX1RPX0NIQVIuZ2V0KGludChpKSwgIiIpIGZvciBpIGluIGlkcykuc3RyaXAoKQoKCmRlZiBncmVlZHlfZGVjb2RlKGxvZ2l0czogbnAubmRhcnJheSkgLT4gc3RyOgogICAgIiIiQ1RDIGdyZWVkeSBkZWNvZGUgb2YgYFtmcmFtZXMsIHZvY2FiXWAgbG9naXRzLiIiIgogICAgYmVzdCA9IGxvZ2l0cy5hcmdtYXgoLTEpCiAgICBvdXQsIHByZXYgPSBbXSwgLTEKICAgIGZvciB0IGluIGJlc3Q6CiAgICAgICAgaWYgdCAhPSBwcmV2IGFuZCB0ICE9IEJMQU5LOgogICAgICAgICAgICBvdXQuYXBwZW5kKGludCh0KSkKICAgICAgICBwcmV2ID0gdAogICAgcmV0dXJuIHJlLnN1YihyIiArIiwgIiAiLCBpZHNfdG9fdGV4dChvdXQpKQoKCmRlZiBhcHBfbm9ybWFsaXplKHg6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJaZXJvIG1lYW4sIHVuaXQgdmFyaWFuY2UsIGV4YWN0bHkgbGlrZSBgQ2h1bmtlZEVtaXNzaW9ucy5ub3JtYWxpemVgIG9uIHRoZSBwaG9uZS4iIiIKICAgIHggPSB4LmFzdHlwZShucC5mbG9hdDY0KQogICAgeCA9IHggLSB4Lm1lYW4oKQogICAgcmV0dXJuICh4IC8gbWF0aC5zcXJ0KHgudmFyKCkgKyAxZS03KSkuYXN0eXBlKG5wLmZsb2F0MzIpCgoKZGVmIGxvYWRfYXVkaW8ocGF0aDogc3RyIHwgUGF0aCwgc3I6IGludCA9IFNBTVBMRV9SQVRFKSAtPiBucC5uZGFycmF5OgogICAgIiIid2F2L2ZsYWMvb2dnL21wMyB2aWEgbGlic25kZmlsZSAoYW55dGhpbmcgZWxzZSB2aWEgZmZtcGVnKSAtPiBtb25vIGZsb2F0MzIgYXQgYHNyYC4iIiIKICAgIGltcG9ydCBzb3VuZGZpbGUgYXMgc2YKICAgIGZyb20gc2NpcHkuc2lnbmFsIGltcG9ydCByZXNhbXBsZV9wb2x5CgogICAgdHJ5OgogICAgICAgIGRhdGEsIGZpbGVfc3IgPSBzZi5yZWFkKHN0cihwYXRoKSwgZHR5cGU9ImZsb2F0MzIiLCBhbHdheXNfMmQ9VHJ1ZSkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgIyBtNGEgLyBhYWMgLyBvcHVzIC8gd2VibSBldGMuOiBkZWNvZGUgd2l0aCBmZm1wZWcgc3RyYWlnaHQgdG8gbW9ubyBhdCBgc3JgLgogICAgICAgIGltcG9ydCBzaHV0aWwsIHN1YnByb2Nlc3MKICAgICAgICBpZiBub3Qgc2h1dGlsLndoaWNoKCJmZm1wZWciKToKICAgICAgICAgICAgcmFpc2UKICAgICAgICByYXcgPSBzdWJwcm9jZXNzLnJ1bihbImZmbXBlZyIsICItdiIsICJlcnJvciIsICItaSIsIHN0cihwYXRoKSwgIi1mIiwgImYzMmxlIiwgIi1hYyIsICIxIiwgIi1hciIsIHN0cihzciksICItIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgY2hlY2s9VHJ1ZSkuc3Rkb3V0CiAgICAgICAgcmV0dXJuIG5wLmZyb21idWZmZXIocmF3LCBkdHlwZT1ucC5mbG9hdDMyKS5jb3B5KCkKICAgIGRhdGEgPSBkYXRhLm1lYW4oYXhpcz0xKQogICAgaWYgZmlsZV9zciAhPSBzcjoKICAgICAgICBnID0gbWF0aC5nY2QoaW50KGZpbGVfc3IpLCBzcikKICAgICAgICBkYXRhID0gcmVzYW1wbGVfcG9seShkYXRhLCBzciAvLyBnLCBpbnQoZmlsZV9zcikgLy8gZykuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICByZXR1cm4gZGF0YQoKCmRlZiBlZGl0X2Rpc3RhbmNlKGEsIGIpIC0+IGludDoKICAgIHByZXYgPSBsaXN0KHJhbmdlKGxlbihiKSArIDEpKQogICAgZm9yIGksIGNhIGluIGVudW1lcmF0ZShhLCAxKToKICAgICAgICBjdXIgPSBbaV0KICAgICAgICBmb3IgaiwgY2IgaW4gZW51bWVyYXRlKGIsIDEpOgogICAgICAgICAgICBjdXIuYXBwZW5kKG1pbihwcmV2W2pdICsgMSwgY3VyW2ogLSAxXSArIDEsIHByZXZbaiAtIDFdICsgKGNhICE9IGNiKSkpCiAgICAgICAgcHJldiA9IGN1cgogICAgcmV0dXJuIHByZXZbLTFdCgoKZGVmIHJlYWRfbWFuaWZlc3QocGF0aDogc3RyIHwgUGF0aCkgLT4gbGlzdFtkaWN0XToKICAgIHdpdGggb3BlbihwYXRoLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIHJldHVybiBbanNvbi5sb2FkcyhsaW5lKSBmb3IgbGluZSBpbiBmIGlmIGxpbmUuc3RyaXAoKV0KCgpkZWYgd3JpdGVfbWFuaWZlc3QocGF0aDogc3RyIHwgUGF0aCwgcm93czogbGlzdFtkaWN0XSkgLT4gTm9uZToKICAgIFBhdGgocGF0aCkucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHdpdGggb3BlbihwYXRoLCAidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgZm9yIHIgaW4gcm93czoKICAgICAgICAgICAgZi53cml0ZShqc29uLmR1bXBzKHIsIGVuc3VyZV9hc2NpaT1GYWxzZSkgKyAiXG4iKQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBDVEMgZm9yY2VkIGFsaWdubWVudCAoVml0ZXJiaSBvdmVyIGJsYW5rLWludGVybGVhdmVkIHN0YXRlcyksIG51bXB5IG9ubHkuCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCmRlZiBjdGNfZm9yY2VkX2FsaWduKGxvZ19wcm9iczogbnAubmRhcnJheSwgdGFyZ2V0czogbGlzdFtpbnRdKSAtPiBucC5uZGFycmF5OgogICAgIiIiUmV0dXJucywgZm9yIGV2ZXJ5IGZyYW1lLCB0aGUgaW5kZXggaW50byBgdGFyZ2V0c2AgaXQgaXMgYWxpZ25lZCB0byAoLTEgPSBibGFuaykuCgogICAgYGxvZ19wcm9ic2AgaXMgYFtmcmFtZXMsIHZvY2FiXWAgbG9nLXNvZnRtYXggb3V0cHV0LiBSYWlzZXMgVmFsdWVFcnJvciB3aGVuIHRoZSBhdWRpbyBpcyB0b28KICAgIHNob3J0IGZvciB0aGUgdHJhbnNjcmlwdC4KICAgICIiIgogICAgVCA9IGxvZ19wcm9icy5zaGFwZVswXQogICAgTCA9IGxlbih0YXJnZXRzKQogICAgUyA9IDIgKiBMICsgMQogICAgbWluX2ZyYW1lcyA9IEwgKyBzdW0oMSBmb3IgaSBpbiByYW5nZSgxLCBMKSBpZiB0YXJnZXRzW2ldID09IHRhcmdldHNbaSAtIDFdKQogICAgaWYgVCA8IG1pbl9mcmFtZXM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIntUfSBmcmFtZXMgY2Fubm90IGhvbGQge0x9IHRva2VucyIpCiAgICB0b2sgPSBucC5mdWxsKFMsIEJMQU5LLCBkdHlwZT1ucC5pbnQ2NCkKICAgIHRva1sxOjoyXSA9IHRhcmdldHMKICAgICMgc2tpcCAocy0yIC0+IHMpIGFsbG93ZWQgZm9yIHRva2VuIHN0YXRlcyB3aG9zZSB0b2tlbiBkaWZmZXJzIGZyb20gdGhlIHByZXZpb3VzIHRva2VuCiAgICBjYW5fc2tpcCA9IG5wLnplcm9zKFMsIGR0eXBlPWJvb2wpCiAgICBmb3IgcyBpbiByYW5nZSgzLCBTLCAyKToKICAgICAgICBjYW5fc2tpcFtzXSA9IHRva1tzXSAhPSB0b2tbcyAtIDJdCgogICAgbmVnID0gLTFlMzAKICAgIHNjb3JlID0gbnAuZnVsbChTLCBuZWcsIGR0eXBlPW5wLmZsb2F0NjQpCiAgICBzY29yZVswXSA9IGxvZ19wcm9ic1swLCB0b2tbMF1dCiAgICBpZiBTID4gMToKICAgICAgICBzY29yZVsxXSA9IGxvZ19wcm9ic1swLCB0b2tbMV1dCiAgICBiYWNrID0gbnAuemVyb3MoKFQsIFMpLCBkdHlwZT1ucC5pbnQ4KSAgIyAwIHN0YXksIDEgZnJvbSBzLTEsIDIgZnJvbSBzLTIKICAgIGZvciB0IGluIHJhbmdlKDEsIFQpOgogICAgICAgIHN0YXkgPSBzY29yZQogICAgICAgIHN0ZXAgPSBucC5jb25jYXRlbmF0ZSgoW25lZ10sIHNjb3JlWzotMV0pKQogICAgICAgIHNraXAgPSBucC53aGVyZShjYW5fc2tpcCwgbnAuY29uY2F0ZW5hdGUoKFtuZWcsIG5lZ10sIHNjb3JlWzotMl0pKSwgbmVnKQogICAgICAgIHN0YWNrZWQgPSBucC5zdGFjayhbc3RheSwgc3RlcCwgc2tpcF0pCiAgICAgICAgY2hvaWNlID0gc3RhY2tlZC5hcmdtYXgoMCkKICAgICAgICBzY29yZSA9IHN0YWNrZWRbY2hvaWNlLCBucC5hcmFuZ2UoUyldICsgbG9nX3Byb2JzW3QsIHRva10KICAgICAgICBiYWNrW3RdID0gY2hvaWNlCiAgICBzID0gUyAtIDEgaWYgUyA9PSAxIG9yIHNjb3JlW1MgLSAxXSA+PSBzY29yZVtTIC0gMl0gZWxzZSBTIC0gMgogICAgcGF0aCA9IG5wLmVtcHR5KFQsIGR0eXBlPW5wLmludDY0KQogICAgZm9yIHQgaW4gcmFuZ2UoVCAtIDEsIC0xLCAtMSk6CiAgICAgICAgcGF0aFt0XSA9IHMKICAgICAgICBzIC09IGludChiYWNrW3QsIHNdKQogICAgcmV0dXJuIG5wLndoZXJlKHBhdGggJSAyID09IDEsIHBhdGggLy8gMiwgLTEpCgoKZGVmIHdvcmRfc3BhbnMoZnJhbWVfdG9rZW46IG5wLm5kYXJyYXksIHdvcmRzOiBsaXN0W3N0cl0pIC0+IGxpc3RbdHVwbGVbaW50LCBpbnRdIHwgTm9uZV06CiAgICAiIiJQZXIgd29yZDogKGZpcnN0IGZyYW1lLCBsYXN0IGZyYW1lKSBvZiBpdHMgbGV0dGVycyBpbiBhIGBjdGNfZm9yY2VkX2FsaWduYCByZXN1bHQuIiIiCiAgICBvd25lciA9IFtdCiAgICBmb3Igd2ksIHcgaW4gZW51bWVyYXRlKHdvcmRzKToKICAgICAgICBpZiB3aToKICAgICAgICAgICAgb3duZXIuYXBwZW5kKC0xKSAgIyB0aGUgd29yZCBzZXBhcmF0b3IgYmVsb25ncyB0byBubyB3b3JkCiAgICAgICAgb3duZXIuZXh0ZW5kKFt3aV0gKiBsZW4odykpCiAgICBzcGFuczogbGlzdFtsaXN0W2ludF0gfCBOb25lXSA9IFtOb25lXSAqIGxlbih3b3JkcykKICAgIGZvciBmLCB0aSBpbiBlbnVtZXJhdGUoZnJhbWVfdG9rZW4pOgogICAgICAgIGlmIHRpIDwgMDoKICAgICAgICAgICAgY29udGludWUKICAgICAgICB3aSA9IG93bmVyW3RpXQogICAgICAgIGlmIHdpIDwgMDoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBzcGFuc1t3aV0gaXMgTm9uZToKICAgICAgICAgICAgc3BhbnNbd2ldID0gW2YsIGZdCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc3BhbnNbd2ldWzFdID0gZgogICAgcmV0dXJuIFt0dXBsZShzKSBpZiBzIGVsc2UgTm9uZSBmb3IgcyBpbiBzcGFuc10K",
"infer.py": "IiIiV2hvbGUtc29uZyBDVEMgZW1pc3Npb25zLCBjb21wdXRlZCBpbiBvdmVybGFwcGluZyBjaHVua3MgdGhlIHdheSB0aGUgcGhvbmUgZG9lcy4KCmBBbGlnbm1lbnRBdWRpby5rdGAgZmVlZHMgdGhlIG1vZGVsIGZpeGVkIGNodW5rcyAoYSBjb3JlIHBsdXMgY29udGV4dCBvbiBlYWNoIHNpZGUpIGFuZCBrZWVwcwpvbmx5IHRoZSBjb3JlIGZyYW1lcy4gRG9pbmcgdGhlIHNhbWUgaGVyZSBtZWFucyBhbGlnbm1lbnQgZXJyb3JzIG1lYXN1cmVkIG9uIHRoZSBkZXNrdG9wIGFyZQp0aGUgZXJyb3JzIHVzZXJzIHdpbGwgc2VlLCBub3QgdGhvc2Ugb2YgYW4gdW5yZWFsaXN0aWNhbGx5IGxvbmcgc2luZ2xlIHBhc3MuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCgpmcm9tIGNvbW1vbiBpbXBvcnQgSE9QLCBTQU1QTEVfUkFURSwgYXBwX25vcm1hbGl6ZQoKCmNsYXNzIEN0Y01vZGVsOgogICAgIiIiQSBQeVRvcmNoIGNoZWNrcG9pbnQgKEhGIGZvbGRlciAvIGh1YiBpZCkgb3IgYW4gZXhwb3J0ZWQgYC5vbm54YCBmaWxlLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBwYXRoOiBzdHIsIGRldmljZTogc3RyIHwgTm9uZSA9IE5vbmUsIHRocmVhZHM6IGludCA9IDQpOgogICAgICAgIHNlbGYucGF0aCA9IHBhdGgKICAgICAgICBpZiBwYXRoLmVuZHN3aXRoKCIub25ueCIpOgogICAgICAgICAgICBpbXBvcnQgb25ueHJ1bnRpbWUgYXMgb3J0CgogICAgICAgICAgICBvcHRzID0gb3J0LlNlc3Npb25PcHRpb25zKCkKICAgICAgICAgICAgb3B0cy5pbnRyYV9vcF9udW1fdGhyZWFkcyA9IHRocmVhZHMKICAgICAgICAgICAgc2VsZi5zZXNzID0gb3J0LkluZmVyZW5jZVNlc3Npb24ocGF0aCwgb3B0cywgcHJvdmlkZXJzPVsiQ1BVRXhlY3V0aW9uUHJvdmlkZXIiXSkKICAgICAgICAgICAgc2VsZi5pbnB1dF9uYW1lID0gc2VsZi5zZXNzLmdldF9pbnB1dHMoKVswXS5uYW1lCiAgICAgICAgICAgIHNlbGYudG9yY2hfbW9kZWwgPSBOb25lCiAgICAgICAgZWxzZToKICAgICAgICAgICAgZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IFdhdjJWZWMyRm9yQ1RDCgogICAgICAgICAgICBzZWxmLmRldmljZSA9IGRldmljZSBvciAoImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKICAgICAgICAgICAgc2VsZi50b3JjaF9tb2RlbCA9IFdhdjJWZWMyRm9yQ1RDLmZyb21fcHJldHJhaW5lZChwYXRoKS50byhzZWxmLmRldmljZSkuZXZhbCgpCiAgICAgICAgICAgIHNlbGYuc2VzcyA9IE5vbmUKCiAgICBkZWYgbG9naXRzKHNlbGYsIHg6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgIiIiYFtzYW1wbGVzXWAgYWxyZWFkeS1ub3JtYWxpc2VkIGZsb2F0MzIgLT4gYFtmcmFtZXMsIHZvY2FiXWAgbG9naXRzLiIiIgogICAgICAgIGlmIHNlbGYuc2VzcyBpcyBub3QgTm9uZToKICAgICAgICAgICAgcmV0dXJuIHNlbGYuc2Vzcy5ydW4oTm9uZSwge3NlbGYuaW5wdXRfbmFtZTogeFtOb25lLCA6XX0pWzBdWzBdCiAgICAgICAgd2l0aCB0b3JjaC5pbmZlcmVuY2VfbW9kZSgpOgogICAgICAgICAgICB0ID0gdG9yY2guZnJvbV9udW1weSh4KVtOb25lXS50byhzZWxmLmRldmljZSkKICAgICAgICAgICAgcmV0dXJuIHNlbGYudG9yY2hfbW9kZWwodCkubG9naXRzWzBdLmZsb2F0KCkuY3B1KCkubnVtcHkoKQoKCmRlZiBzb25nX2xvZ19wcm9icyhtb2RlbDogQ3RjTW9kZWwsIGF1ZGlvOiBucC5uZGFycmF5LCBjb3JlX3M6IGZsb2F0ID0gMjAuMCwgY29udGV4dF9zOiBmbG9hdCA9IDIuMCkgLT4gbnAubmRhcnJheToKICAgICIiImBbZnJhbWVzLCB2b2NhYl1gIGxvZy1wcm9iYWJpbGl0aWVzIGZvciBhIHdob2xlIHNvbmcsIG9uZSBmcmFtZSBwZXIgMjAgbXMuIiIiCiAgICBjb3JlID0gaW50KGNvcmVfcyAqIFNBTVBMRV9SQVRFKSAvLyBIT1AgKiBIT1AKICAgIGN0eCA9IGludChjb250ZXh0X3MgKiBTQU1QTEVfUkFURSkgLy8gSE9QICogSE9QCiAgICB0b3RhbF9mcmFtZXMgPSBsZW4oYXVkaW8pIC8vIEhPUAogICAgb3V0ID0gW10KICAgIHN0YXJ0ID0gMAogICAgd2hpbGUgc3RhcnQgPCBsZW4oYXVkaW8pOgogICAgICAgIGEgPSBtYXgoMCwgc3RhcnQgLSBjdHgpCiAgICAgICAgYiA9IG1pbihsZW4oYXVkaW8pLCBzdGFydCArIGNvcmUgKyBjdHgpCiAgICAgICAgY2h1bmsgPSBhdWRpb1thOmJdCiAgICAgICAgaWYgbGVuKGNodW5rKSA8IEhPUCAqIDI6CiAgICAgICAgICAgIGJyZWFrCiAgICAgICAgbGcgPSBtb2RlbC5sb2dpdHMoYXBwX25vcm1hbGl6ZShjaHVuaykpCiAgICAgICAgc2tpcCA9IChzdGFydCAtIGEpIC8vIEhPUAogICAgICAgIGtlZXAgPSBtaW4oY29yZSAvLyBIT1AsIHRvdGFsX2ZyYW1lcyAtIHN0YXJ0IC8vIEhPUCkKICAgICAgICBvdXQuYXBwZW5kKGxnW3NraXA6IHNraXAgKyBrZWVwXSkKICAgICAgICBzdGFydCArPSBjb3JlCiAgICBsZyA9IG5wLmNvbmNhdGVuYXRlKG91dCwgMClbOnRvdGFsX2ZyYW1lc10KICAgIGxnID0gbGcgLSBsZy5tYXgoLTEsIGtlZXBkaW1zPVRydWUpCiAgICByZXR1cm4gbGcgLSBucC5sb2cobnAuZXhwKGxnKS5zdW0oLTEsIGtlZXBkaW1zPVRydWUpKQo=",
"prepare_data.py": "...",
"train.py": "...",
"evaluate_alignment.py": "IiIiVGhlIG51bWJlciB0aGF0IG1hdHRlcnM6IGhvdyBmYXIgYXJlIHdvcmQgc3RhcnRzIGZyb20gd2hlcmUgdGhlIHNpbmdlciBhY3R1YWxseSBzaW5ncyB0aGVtPwoKUnVucyBrbm93bi1seXJpY3MgQ1RDIGFsaWdubWVudCBvdmVyIHdob2xlIHNvbmdzIChmdWxsIG1peGVzLCBjaHVua2VkIGxpa2UgdGhlIHBob25lKSBhbmQKY29tcGFyZXMgZXZlcnkgd29yZCdzIHByZWRpY3RlZCBzdGFydCB3aXRoIGEgaHVtYW4gYW5ub3RhdGlvbi4gVXNlIEphbWVuZG9MeXJpY3MKKGh0dHBzOi8vZ2l0aHViLmNvbS9mOTAvamFtZW5kb2x5cmljcyksIEVuZ2xpc2ggc29uZ3Mgb25seSwgYW5kIGtlZXAgdGhlc2Ugc29uZ3Mgb3V0IG9mIHRyYWluaW5nLgoKICAgIHB5dGhvbiBldmFsdWF0ZV9hbGlnbm1lbnQucHkgLS1qYW1lbmRvIGRhdGEvamFtZW5kb2x5cmljcyBcCiAgICAgICAgLS1tb2RlbHMgZmFjZWJvb2svd2F2MnZlYzItYmFzZS05NjBoIHJ1bnMvc2luZ2luZy12MS9iZXN0IHJ1bnMvc2luZ2luZy12MS9tb2RlbF9pbnQ4Lm9ubngKCkV4cGVjdGVkIGxheW91dCAodGhlIHJlcG8ncyBvd24pOiBtcDMvPHNvbmc+Lm1wMywgbHlyaWNzLzxzb25nPi53b3Jkcy50eHQgKG9yIC50eHQpIGFuZAphbm5vdGF0aW9ucy93b3Jkcy88c29uZz4uY3N2IHdob3NlIGZpcnN0IGNvbHVtbiBpcyBlYWNoIHdvcmQncyBzdGFydCBpbiBzZWNvbmRzLiBBCkphbWVuZG9MeXJpY3MuY3N2IHdpdGggYSBMYW5ndWFnZSBjb2x1bW4sIGlmIHByZXNlbnQsIGlzIHVzZWQgdG8ga2VlcCBFbmdsaXNoIHNvbmdzIG9ubHkuCkFueSBvdGhlciBkYXRhc2V0IGluIHRoYXQgbGF5b3V0IHdvcmtzIHRvby4KCldyaXRlcyBhIEpTT04gcmVwb3J0IChtZWRpYW4vbWVhbiBlcnJvciwgc2hhcmUgd2l0aGluIDEwMC8yMDAvMzAwIG1zLCBwZXIgc29uZykgbmV4dCB0byBpdC4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgY3N2CmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCgpmcm9tIGNvbW1vbiBpbXBvcnQgSE9QLCBTQU1QTEVfUkFURSwgY3RjX2ZvcmNlZF9hbGlnbiwgbG9hZF9hdWRpbywgbm9ybWFsaXplX3dvcmRzLCB3b3JkX3NwYW5zLCB3b3Jkc190b19pZHMKZnJvbSBpbmZlciBpbXBvcnQgQ3RjTW9kZWwsIHNvbmdfbG9nX3Byb2JzCgoKZGVmIGxvYWRfc29uZ3Mocm9vdDogUGF0aCwgZW5nbGlzaF9vbmx5OiBib29sKToKICAgIGtlZXAgPSBOb25lCiAgICBtZXRhID0gcm9vdCAvICJKYW1lbmRvTHlyaWNzLmNzdiIKICAgIGlmIGVuZ2xpc2hfb25seSBhbmQgbWV0YS5leGlzdHMoKToKICAgICAgICB3aXRoIG9wZW4obWV0YSwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICAgICAgcm93cyA9IGxpc3QoY3N2LkRpY3RSZWFkZXIoZikpCiAgICAgICAgbGFuZ19rZXkgPSBuZXh0KChrIGZvciBrIGluIHJvd3NbMF0gaWYgay5sb3dlcigpID09ICJsYW5ndWFnZSIpLCBOb25lKSBpZiByb3dzIGVsc2UgTm9uZQogICAgICAgIG5hbWVfa2V5ID0gbmV4dCgoayBmb3IgayBpbiByb3dzWzBdIGlmIGsubG93ZXIoKSBpbiB7ImZpbGVwYXRoIiwgIm5hbWUiLCAiZmlsZSJ9KSwgTm9uZSkgaWYgcm93cyBlbHNlIE5vbmUKICAgICAgICBpZiBsYW5nX2tleSBhbmQgbmFtZV9rZXk6CiAgICAgICAgICAgIGtlZXAgPSB7UGF0aChyW25hbWVfa2V5XSkuc3RlbSBmb3IgciBpbiByb3dzIGlmIHJbbGFuZ19rZXldLmxvd2VyKCkuc3RhcnRzd2l0aCgiZW4iKX0KICAgIGF1ZGlvX2RpciA9IHJvb3QgLyAibXAzIiBpZiAocm9vdCAvICJtcDMiKS5leGlzdHMoKSBlbHNlIHJvb3QgLyAiYXVkaW8iCiAgICBmb3IgYXVkaW8gaW4gc29ydGVkKGF1ZGlvX2Rpci5nbG9iKCIqIikpOgogICAgICAgIGlmIGtlZXAgaXMgbm90IE5vbmUgYW5kIGF1ZGlvLnN0ZW0gbm90IGluIGtlZXA6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgd29yZHNfZmlsZSA9IG5leHQoKHAgZm9yIHAgaW4gKHJvb3QgLyAibHlyaWNzIiAvIGYie2F1ZGlvLnN0ZW19LndvcmRzLnR4dCIsIHJvb3QgLyAibHlyaWNzIiAvIGYie2F1ZGlvLnN0ZW19LnR4dCIpCiAgICAgICAgICAgICAgICAgICAgICAgICAgIGlmIHAuZXhpc3RzKCkpLCBOb25lKQogICAgICAgIGFubiA9IHJvb3QgLyAiYW5ub3RhdGlvbnMiIC8gIndvcmRzIiAvIGYie2F1ZGlvLnN0ZW19LmNzdiIKICAgICAgICBpZiB3b3Jkc19maWxlIGlzIE5vbmUgb3Igbm90IGFubi5leGlzdHMoKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICByYXdfd29yZHMgPSB3b3Jkc19maWxlLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKS5zcGxpdCgpCiAgICAgICAgc3RhcnRzID0gW10KICAgICAgICB3aXRoIG9wZW4oYW5uLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgICAgICBmb3Igcm93IGluIGNzdi5yZWFkZXIoZik6CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgc3RhcnRzLmFwcGVuZChmbG9hdChyb3dbMF0pKQogICAgICAgICAgICAgICAgZXhjZXB0IChWYWx1ZUVycm9yLCBJbmRleEVycm9yKToKICAgICAgICAgICAgICAgICAgICBjb250aW51ZSAgIyBoZWFkZXIKICAgICAgICBpZiBsZW4oc3RhcnRzKSAhPSBsZW4ocmF3X3dvcmRzKToKICAgICAgICAgICAgcHJpbnQoZiJza2lwIHthdWRpby5zdGVtfToge2xlbihyYXdfd29yZHMpfSB3b3JkcyB2cyB7bGVuKHN0YXJ0cyl9IGFubm90YXRpb25zIikKICAgICAgICAgICAgY29udGludWUKICAgICAgICB5aWVsZCBhdWRpbywgcmF3X3dvcmRzLCBzdGFydHMKCgpkZWYgZXZhbHVhdGVfbW9kZWwobW9kZWxfcGF0aDogc3RyLCBzb25ncywgdGhyZWFkczogaW50KToKICAgIG1vZGVsID0gQ3RjTW9kZWwobW9kZWxfcGF0aCwgdGhyZWFkcz10aHJlYWRzKQogICAgZXJyb3JzLCBwZXJfc29uZyA9IFtdLCB7fQogICAgZm9yIGF1ZGlvX3BhdGgsIHJhd193b3Jkcywgc3RhcnRzIGluIHNvbmdzOgogICAgICAgIGF1ZGlvID0gbG9hZF9hdWRpbyhhdWRpb19wYXRoKQogICAgICAgIGxwID0gc29uZ19sb2dfcHJvYnMobW9kZWwsIGF1ZGlvKQogICAgICAgICMga2VlcCB0aGUgZGlzcGxheS13b3JkIDwtPiBhbm5vdGF0aW9uIHBhaXJpbmcgd2hpbGUgbm9ybWFsaXNpbmcKICAgICAgICBwYWlycyA9IFtdCiAgICAgICAgZm9yIHcsIHMgaW4gemlwKHJhd193b3Jkcywgc3RhcnRzKToKICAgICAgICAgICAgbm9ybSA9IG5vcm1hbGl6ZV93b3Jkcyh3KQogICAgICAgICAgICAjIE9uZSBhbm5vdGF0aW9uIHBlciBkaXNwbGF5IHdvcmQ6IHNraXAgd29yZHMgdGhhdCBub3JtYWxpc2UgdG8gbm90aGluZywgdG8gc2V2ZXJhbAogICAgICAgICAgICAjIHdvcmRzICgicm9jay1uLXJvbGwiKSBvciBjb250YWluIGRpZ2l0cywgYXMgdGhlIGFwcCBpbnRlcnBvbGF0ZXMgdGhvc2UgYW55d2F5LgogICAgICAgICAgICBpZiBsZW4obm9ybSkgPT0gMSBhbmQgbm90IGFueShjLmlzZGlnaXQoKSBmb3IgYyBpbiBub3JtWzBdKToKICAgICAgICAgICAgICAgIHBhaXJzLmFwcGVuZCgobm9ybVswXSwgcykpCiAgICAgICAgd29yZHMgPSBbdyBmb3IgdywgXyBpbiBwYWlyc10KICAgICAgICB0cnk6CiAgICAgICAgICAgIHNwYW5zID0gd29yZF9zcGFucyhjdGNfZm9yY2VkX2FsaWduKGxwLCB3b3Jkc190b19pZHMod29yZHMpKSwgd29yZHMpCiAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3I6CiAgICAgICAgICAgIHByaW50KGYie2F1ZGlvX3BhdGguc3RlbX06IGFsaWdubWVudCBmYWlsZWQiKQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGUgPSBbYWJzKHNwWzBdICogSE9QIC8gU0FNUExFX1JBVEUgLSBzKSBmb3Igc3AsIChfLCBzKSBpbiB6aXAoc3BhbnMsIHBhaXJzKSBpZiBzcCBpcyBub3QgTm9uZV0KICAgICAgICBlcnJvcnMgKz0gZQogICAgICAgIHBlcl9zb25nW2F1ZGlvX3BhdGguc3RlbV0gPSB7IndvcmRzIjogbGVuKGUpLCAibWVkaWFuX3MiOiBmbG9hdChucC5tZWRpYW4oZSkpIGlmIGUgZWxzZSBOb25lLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIndpdGhpbl8zMDBtcyI6IGZsb2F0KG5wLm1lYW4obnAuYXJyYXkoZSkgPD0gMC4zKSkgaWYgZSBlbHNlIE5vbmV9CiAgICAgICAgcHJpbnQoZiIgIHthdWRpb19wYXRoLnN0ZW19OiBtZWRpYW4ge3Blcl9zb25nW2F1ZGlvX3BhdGguc3RlbV1bJ21lZGlhbl9zJ106LjNmfXMgb3ZlciB7bGVuKGUpfSB3b3JkcyIpCiAgICBlID0gbnAuYXJyYXkoZXJyb3JzKQogICAgcmV0dXJuIHsKICAgICAgICAibW9kZWwiOiBtb2RlbF9wYXRoLAogICAgICAgICJ3b3JkcyI6IGludChsZW4oZSkpLAogICAgICAgICJtZWRpYW5fcyI6IGZsb2F0KG5wLm1lZGlhbihlKSkgaWYgbGVuKGUpIGVsc2UgTm9uZSwKICAgICAgICAibWVhbl9zIjogZmxvYXQoZS5tZWFuKCkpIGlmIGxlbihlKSBlbHNlIE5vbmUsCiAgICAgICAgInA5NV9zIjogZmxvYXQobnAucGVyY2VudGlsZShlLCA5NSkpIGlmIGxlbihlKSBlbHNlIE5vbmUsCiAgICAgICAgIndpdGhpbl8xMDBtcyI6IGZsb2F0KChlIDw9IDAuMSkubWVhbigpKSBpZiBsZW4oZSkgZWxzZSBOb25lLAogICAgICAgICJ3aXRoaW5fMjAwbXMiOiBmbG9hdCgoZSA8PSAwLjIpLm1lYW4oKSkgaWYgbGVuKGUpIGVsc2UgTm9uZSwKICAgICAgICAid2l0aGluXzMwMG1zIjogZmxvYXQoKGUgPD0gMC4zKS5tZWFuKCkpIGlmIGxlbihlKSBlbHNlIE5vbmUsCiAgICAgICAgInNvbmdzIjogcGVyX3NvbmcsCiAgICB9CgoKZGVmIG1haW4oKToKICAgIHAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fLCBmb3JtYXR0ZXJfY2xhc3M9YXJncGFyc2UuUmF3RGVzY3JpcHRpb25IZWxwRm9ybWF0dGVyKQogICAgcC5hZGRfYXJndW1lbnQoIi0tamFtZW5kbyIsIHJlcXVpcmVkPVRydWUpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1tb2RlbHMiLCBuYXJncz0iKyIsIHJlcXVpcmVkPVRydWUpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1hbGwtbGFuZ3VhZ2VzIiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLXRocmVhZHMiLCB0eXBlPWludCwgZGVmYXVsdD00KQogICAgcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgZGVmYXVsdD0iYWxpZ25tZW50X3JlcG9ydC5qc29uIikKICAgIGEgPSBwLnBhcnNlX2FyZ3MoKQogICAgc29uZ3MgPSBsaXN0KGxvYWRfc29uZ3MoUGF0aChhLmphbWVuZG8pLCBub3QgYS5hbGxfbGFuZ3VhZ2VzKSkKICAgIGlmIG5vdCBzb25nczoKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KCJubyBzb25ncyBmb3VuZDsgY2hlY2sgdGhlIGZvbGRlciBsYXlvdXQgZGVzY3JpYmVkIGluIC0taGVscCIpCiAgICBwcmludChmIntsZW4oc29uZ3MpfSBzb25ncyIpCiAgICByZXN1bHRzID0gW10KICAgIGZvciBtIGluIGEubW9kZWxzOgogICAgICAgIHByaW50KGYiPT0ge219IikKICAgICAgICByID0gZXZhbHVhdGVfbW9kZWwobSwgc29uZ3MsIGEudGhyZWFkcykKICAgICAgICByZXN1bHRzLmFwcGVuZChyKQogICAgICAgIHByaW50KHtrOiB2IGZvciBrLCB2IGluIHIuaXRlbXMoKSBpZiBrICE9ICJzb25ncyJ9KQogICAgUGF0aChhLm91dCkud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlc3VsdHMsIGluZGVudD0yKSkKICAgIHByaW50KCJcbm1vZGVsIi5sanVzdCg2MCksICJtZWRpYW4gIHdpdGhpbjIwMG1zICB3aXRoaW4zMDBtcyIpCiAgICBmb3IgciBpbiByZXN1bHRzOgogICAgICAgIHByaW50KHJbIm1vZGVsIl1bLTU4Ol0ubGp1c3QoNjApLCBmIntyWydtZWRpYW5fcyddOi4zZn1zICB7clsnd2l0aGluXzIwMG1zJ106LjElfSAgICAgICB7clsnd2l0aGluXzMwMG1zJ106LjElfSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=",
"export_onnx.py": "IiIiRXhwb3J0IGEgZmluZS10dW5lZCBjaGVja3BvaW50IHRvIHRoZSBpbnQ4IE9OTlggZmlsZSB0aGUgYXBwIGRvd25sb2Fkcy4KCiAgICBweXRob24gZXhwb3J0X29ubngucHkgLS1jaGVja3BvaW50IHJ1bnMvc2luZ2luZy12MS9iZXN0IC0tb3V0IHJ1bnMvc2luZ2luZy12MS9tb2RlbF9pbnQ4Lm9ubngKCkNoZWNrcyB0aGF0IHRoZSBleHBvcnRlZCBtb2RlbCBoYXMgdGhlIHNhbWUgaW5wdXQvb3V0cHV0IGNvbnRyYWN0IGFzIHRoZSBjdXJyZW50IG9uZQooYGlucHV0X3ZhbHVlc2AgWzEsIHNhbXBsZXNdIC0+IGBsb2dpdHNgIFsxLCBmcmFtZXMsIDMyXSksIGNvbXBhcmVzIGl0IHdpdGggUHlUb3JjaCBvbiBhIHJlYWwgb3IKc3ludGhldGljIGNsaXAsIHRoZW4gcHJpbnRzIHRoZSBTSEEtMjU2IGFuZCB0aGUgS290bGluIGNvbnN0YW50cyBmb3IgYEFsaWdubWVudE1vZGVsTWFuYWdlcmAuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGhhc2hsaWIKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCgpmcm9tIGNvbW1vbiBpbXBvcnQgU0FNUExFX1JBVEUsIFZPQ0FCX1NJWkUsIGFwcF9ub3JtYWxpemUsIGdyZWVkeV9kZWNvZGUsIGxvYWRfYXVkaW8KCgpkZWYgc2hhMjU2KHBhdGg6IFBhdGgpIC0+IHN0cjoKICAgIGggPSBoYXNobGliLnNoYTI1NigpCiAgICB3aXRoIG9wZW4ocGF0aCwgInJiIikgYXMgZjoKICAgICAgICBmb3IgYmxvY2sgaW4gaXRlcihsYW1iZGE6IGYucmVhZCgxIDw8IDIwKSwgYiIiKToKICAgICAgICAgICAgaC51cGRhdGUoYmxvY2spCiAgICByZXR1cm4gaC5oZXhkaWdlc3QoKQoKCmRlZiBtYWluKCk6CiAgICBwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXywgZm9ybWF0dGVyX2NsYXNzPWFyZ3BhcnNlLlJhd0Rlc2NyaXB0aW9uSGVscEZvcm1hdHRlcikKICAgIHAuYWRkX2FyZ3VtZW50KCItLWNoZWNrcG9pbnQiLCByZXF1aXJlZD1UcnVlKQogICAgcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHAuYWRkX2FyZ3VtZW50KCItLXRlc3QtYXVkaW8iLCBkZWZhdWx0PU5vbmUsIGhlbHA9ImEgc3VuZyBjbGlwIHRvIGNvbXBhcmUgUHlUb3JjaCB2cyBPTk5YIG9uIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLXF1YW50aXplLWNvbnYiLCBhY3Rpb249InN0b3JlX3RydWUiLAogICAgICAgICAgICAgICAgICAgaGVscD0iYWxzbyBxdWFudGlzZSB0aGUgY29udm9sdXRpb25hbCBmZWF0dXJlIGVuY29kZXIgKHNtYWxsZXIgZmlsZSwgfjk1IE1CIGxpa2UgdGhlIGN1cnJlbnQgIgogICAgICAgICAgICAgICAgICAgICAgICAibW9kZWwsIGF0IHNvbWUgYWNjdXJhY3kgY29zdCkuIERlZmF1bHQgcXVhbnRpc2VzIG9ubHkgdGhlIHRyYW5zZm9ybWVyJ3MgbGluZWFyIGxheWVycy4iKQogICAgcC5hZGRfYXJndW1lbnQoIi0tdmVyc2lvbiIsIGRlZmF1bHQ9IncydjItYmFzZS05NjBoLXNpbmdpbmctaW50OC12MSIpCiAgICBhID0gcC5wYXJzZV9hcmdzKCkKCiAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgV2F2MlZlYzJGb3JDVEMKICAgIGZyb20gb25ueHJ1bnRpbWUucXVhbnRpemF0aW9uIGltcG9ydCBRdWFudFR5cGUsIHF1YW50aXplX2R5bmFtaWMKICAgIGltcG9ydCBvbm54cnVudGltZSBhcyBvcnQKCiAgICBtb2RlbCA9IFdhdjJWZWMyRm9yQ1RDLmZyb21fcHJldHJhaW5lZChhLmNoZWNrcG9pbnQpLmV2YWwoKQogICAgYXNzZXJ0IG1vZGVsLmNvbmZpZy52b2NhYl9zaXplID09IFZPQ0FCX1NJWkUKICAgIG91dCA9IFBhdGgoYS5vdXQpCiAgICBvdXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGZwMzIgPSBvdXQud2l0aF9uYW1lKG91dC5zdGVtICsgIl9mcDMyLm9ubngiKQogICAgZHVtbXkgPSB0b3JjaC5yYW5kbigxLCBTQU1QTEVfUkFURSAqIDQpCiAgICBrd2FyZ3MgPSBkaWN0KGlucHV0X25hbWVzPVsiaW5wdXRfdmFsdWVzIl0sIG91dHB1dF9uYW1lcz1bImxvZ2l0cyJdLAogICAgICAgICAgICAgICAgICBkeW5hbWljX2F4ZXM9eyJpbnB1dF92YWx1ZXMiOiB7MDogImJhdGNoX3NpemUiLCAxOiAic2VxdWVuY2VfbGVuZ3RoIn0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImxvZ2l0cyI6IHswOiAiYmF0Y2hfc2l6ZSIsIDE6ICJmcmFtZXMifX0sCiAgICAgICAgICAgICAgICAgIG9wc2V0X3ZlcnNpb249MTQsIGRvX2NvbnN0YW50X2ZvbGRpbmc9VHJ1ZSkKICAgIHRyeToKICAgICAgICB0b3JjaC5vbm54LmV4cG9ydChtb2RlbCwgKGR1bW15LCksIHN0cihmcDMyKSwgZHluYW1vPUZhbHNlLCAqKmt3YXJncykKICAgIGV4Y2VwdCBUeXBlRXJyb3I6ICAjIG9sZGVyIHRvcmNoIHdpdGhvdXQgdGhlIGR5bmFtbyBzd2l0Y2gKICAgICAgICB0b3JjaC5vbm54LmV4cG9ydChtb2RlbCwgKGR1bW15LCksIHN0cihmcDMyKSwgKiprd2FyZ3MpCgogICAgb3BzID0gTm9uZSBpZiBhLnF1YW50aXplX2NvbnYgZWxzZSBbIk1hdE11bCIsICJHZW1tIl0KICAgIHF1YW50aXplX2R5bmFtaWMoc3RyKGZwMzIpLCBzdHIob3V0KSwgd2VpZ2h0X3R5cGU9UXVhbnRUeXBlLlFVSW50OCwgb3BfdHlwZXNfdG9fcXVhbnRpemU9b3BzKQoKICAgIHggPSBsb2FkX2F1ZGlvKGEudGVzdF9hdWRpbykgaWYgYS50ZXN0X2F1ZGlvIGVsc2UgbnAucmFuZG9tLnJhbmRuKFNBTVBMRV9SQVRFICogNikuYXN0eXBlKG5wLmZsb2F0MzIpICogMC4xCiAgICB4ID0gYXBwX25vcm1hbGl6ZSh4KQogICAgd2l0aCB0b3JjaC5pbmZlcmVuY2VfbW9kZSgpOgogICAgICAgIHJlZiA9IG1vZGVsKHRvcmNoLmZyb21fbnVtcHkoeClbTm9uZV0pLmxvZ2l0c1swXS5udW1weSgpCiAgICBzZXNzID0gb3J0LkluZmVyZW5jZVNlc3Npb24oc3RyKG91dCksIHByb3ZpZGVycz1bIkNQVUV4ZWN1dGlvblByb3ZpZGVyIl0pCiAgICBpLCBvID0gc2Vzcy5nZXRfaW5wdXRzKClbMF0sIHNlc3MuZ2V0X291dHB1dHMoKVswXQogICAgZ290ID0gc2Vzcy5ydW4oTm9uZSwge2kubmFtZTogeFtOb25lXX0pWzBdWzBdCiAgICBhc3NlcnQgZ290LnNoYXBlID09IHJlZi5zaGFwZSBhbmQgZ290LnNoYXBlWzFdID09IFZPQ0FCX1NJWkUsIChnb3Quc2hhcGUsIHJlZi5zaGFwZSkKICAgIGFncmVlID0gZmxvYXQoKGdvdC5hcmdtYXgoLTEpID09IHJlZi5hcmdtYXgoLTEpKS5tZWFuKCkpCiAgICBwcmludChmImlucHV0IHtpLm5hbWV9IHtpLnNoYXBlfSAtPiBvdXRwdXQge28ubmFtZX0ge28uc2hhcGV9IikKICAgIHByaW50KGYiZnJhbWUgYXJnbWF4IGFncmVlbWVudCBpbnQ4IHZzIGZwMzI6IHthZ3JlZTouMSV9IikKICAgIGlmIGEudGVzdF9hdWRpbzoKICAgICAgICBwcmludCgicHl0b3JjaDoiLCBncmVlZHlfZGVjb2RlKHJlZikpCiAgICAgICAgcHJpbnQoIm9ubnggICA6IiwgZ3JlZWR5X2RlY29kZShnb3QpKQogICAgaWYgYWdyZWUgPCAwLjk6CiAgICAgICAgcHJpbnQoIldBUk5JTkc6IHF1YW50aXNhdGlvbiBjaGFuZ2VkIG1hbnkgZnJhbWVzOyBldmFsdWF0ZV9hbGlnbm1lbnQucHkgdGhlIC5vbm54IGJlZm9yZSBzaGlwcGluZyBpdC4iKQoKICAgIGRpZ2VzdCA9IHNoYTI1NihvdXQpCiAgICBzaXplID0gb3V0LnN0YXQoKS5zdF9zaXplCiAgICBwcmludChmIlxue291dH0gIHtzaXplLzFlNjouMWZ9IE1CICBzaGEyNTYge2RpZ2VzdH0iKQogICAgcHJpbnQoIlxuVXBsb2FkIHRoZSBmaWxlIHNvbWV3aGVyZSBpdCBjYW4gYmUgZG93bmxvYWRlZCBvdmVyIEhUVFBTIChhIEh1Z2dpbmcgRmFjZSBtb2RlbCByZXBvIG9yIGEgR2l0SHViIikKICAgIHByaW50KCJyZWxlYXNlKSwgdGhlbiBpbiBBbGlnbm1lbnRNb2RlbE1hbmFnZXIua3Qgc2V0OlxuIikKICAgIHByaW50KGYnICAgICAgICBjb25zdCB2YWwgTU9ERUxfVkVSU0lPTiA9ICJ7YS52ZXJzaW9ufSInKQogICAgcHJpbnQoZicgICAgICAgIHByaXZhdGUgY29uc3QgdmFsIE1PREVMX0ZJTEUgPSAie2EudmVyc2lvbn0ub25ueCInKQogICAgcHJpbnQoJyAgICAgICAgcHJpdmF0ZSBjb25zdCB2YWwgTU9ERUxfVVJMID0gImh0dHBzOi8vPHdoZXJlIHlvdSB1cGxvYWRlZCBpdD4iJykKICAgIHByaW50KGYnICAgICAgICBwcml2YXRlIGNvbnN0IHZhbCBNT0RFTF9TSEEyNTYgPSAie2RpZ2VzdH0iJykKICAgIHByaW50KGYiICAgICAgICBjb25zdCB2YWwgTU9ERUxfQllURVNfQVBQUk9YID0ge3JvdW5kKHNpemUgLyAxMDQ4NTc2KX1MICogMTAyNCAqIDEwMjQiKQogICAgaWYgc2l6ZSA+IDE2MCAqIDEwMjQgKiAxMDI0OgogICAgICAgIHByaW50KCJcbkFsc28gcmFpc2UgTU9ERUxfQllURVNfTUFYIChjdXJyZW50bHkgMTYwIE1CKTogdGhpcyBmaWxlIGlzIGJpZ2dlci4iKQogICAgZnAzMi51bmxpbmsobWlzc2luZ19vaz1UcnVlKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK",
"requirements.txt": "IyBJbnN0YWxsIFB5VG9yY2ggZm9yIHlvdXIgR1BVIGZpcnN0IChodHRwczovL3B5dG9yY2gub3JnL2dldC1zdGFydGVkL2xvY2FsbHkvKSwgdGhlbjoKdG9yY2g+PTIuMgp0cmFuc2Zvcm1lcnM+PTQuNDAsPDUKbnVtcHk+PTEuMjQKc2NpcHk+PTEuMTAKc291bmRmaWxlPj0wLjEyICAgIyBsaWJzbmRmaWxlID49IDEuMSByZWFkcyBtcDMgKG5lZWRlZCBmb3IgSmFtZW5kb0x5cmljcykKb25ueD49MS4xNQpvbm54cnVudGltZT49MS4xNwo="
}
for name, b64 in FILES.items():
    open(os.path.join(KIT, name), 'wb').write(base64.b64decode(b64))
os.chdir(KIT)
print(sorted(os.listdir(KIT)))

In [ ]:
!pip -q install "transformers>=4.40,<5" "soundfile>=0.12" onnx onnxruntime musdb huggingface_hub remotezip

## Data: real songs first (downloads itself, 30-60 min)
Every source is optional: if one fails, it says why and the rest carry on.

| Source | What | Used for |
|---|---|---|
| **MUSDB18-HQ + lyrics extension** | 141 real, professionally produced songs (rock, pop, rap, electronic...), full mix *and* isolated vocals, lyrics in 3-12 s sections | training (real songs) |
| **Your library** (optional) | your own songs + synced `.lrc` lyrics | training (closest to what you listen to) |
| GTSinger English | studio singing with word labels | training |
| Children's Song Dataset | 50 sung songs + lyrics | training |
| MUSDB18 accompaniment | the instrumental part of each song (mix minus vocals) | mixed under clean vocals |
| **JamendoLyrics** | real songs with human word times | **evaluation only, never trained on** |

In [ ]:
import os, glob, zipfile, subprocess, traceback, math
import numpy as np, soundfile as sf
from scipy.signal import resample_poly
STATUS = {}
def step(name, fn):
    try:
        fn(); STATUS[name] = "ok"
    except Exception as e:
        traceback.print_exc(); STATUS[name] = f"FAILED: {e}"
    print(f"== {name}: {STATUS[name]}")

# 0. Data already prepared by kaggle_data_prep.ipynb (or a run that timed out)? Use it instead of
#    downloading again: *.tar packs and/or an unpacked pp_stage folder.
import tarfile, shutil
def restore_prepared():
    found = 0
    for root in PREPARED_SOURCES:
        for tar in sorted(glob.glob(f"{root}/**/pp_data_pack/*.tar", recursive=True)):
            name = os.path.basename(tar)[:-4]
            if os.path.exists(f"{DATA}/.restored_{name}"): continue
            print("restoring", tar)
            with tarfile.open(tar) as t: t.extractall(DATA)
            open(f"{DATA}/.restored_{name}", "w").close(); found += 1
        for stage in glob.glob(f"{root}/**/pp_stage", recursive=True):
            if os.path.abspath(stage) == os.path.abspath(DATA): continue
            print("restoring", stage)
            shutil.copytree(stage, DATA, dirs_exist_ok=True); found += 1
    print("prepared data restored from", found, "source(s)" if found else "- none attached, downloading")
restore_prepared()

def to16k_mono(x, sr):
    x = x.mean(axis=1) if x.ndim == 2 else x
    g = math.gcd(int(sr), 16000)
    return resample_poly(x, 16000 // g, int(sr) // g).astype(np.float32)

# 1. MUSDB18-HQ: fetch only mixture + vocals of each song straight out of the 22.7 GB zip
#    (HTTP range requests), shrink to 16 kHz mono, and derive the accompaniment as mix - vocals.
MUS = f"{DATA}/musdb16k"; ACC = f"{DATA}/accompaniment"
def get_musdb():
    # Zenodo sometimes drops a connection mid-file. Each song is retried (re-opening the zip)
    # and a song that keeps failing is skipped, so one bad moment no longer loses all 150.
    import time
    from remotezip import RemoteZip
    URL = "https://zenodo.org/records/3338373/files/musdb18hq.zip?download=1"
    os.makedirs(ACC, exist_ok=True)
    have = len(glob.glob(f"{MUS}/*/*/vocals.wav"))
    if have >= 140:
        print(f"  {have} songs already prepared - no download needed"); return
    def open_zip():
        for attempt in range(6):
            try:
                return RemoteZip(URL)
            except Exception as e:
                print(f"  opening the zip failed ({e}); retrying in {15 * (attempt + 1)} s")
                time.sleep(15 * (attempt + 1))
        raise RuntimeError("could not open the MUSDB18-HQ zip on Zenodo")
    z = open_zip()
    names = [i.filename for i in z.infolist() if i.filename.endswith("/mixture.wav")]
    done = {os.path.basename(os.path.dirname(p)) for p in glob.glob(f"{MUS}/*/*/vocals.wav")}
    failed = []
    for k, mix_name in enumerate(names):
        track_dir = os.path.dirname(mix_name); track = os.path.basename(track_dir)
        if track in done: continue
        split = "test" if "/test/" in f"/{mix_name}" else "train"
        out = f"{MUS}/{split}/{track}"
        tmp = f"{DATA}/_tmp"
        for attempt in range(5):
            try:
                subprocess.run(["rm", "-rf", tmp]); os.makedirs(tmp, exist_ok=True)
                z.extract(mix_name, tmp); z.extract(f"{track_dir}/vocals.wav", tmp)
                mix, sr = sf.read(f"{tmp}/{mix_name}", dtype="float32"); voc, _ = sf.read(f"{tmp}/{track_dir}/vocals.wav", dtype="float32")
                m16, v16 = to16k_mono(mix, sr), to16k_mono(voc, sr)
                os.makedirs(out, exist_ok=True)
                sf.write(f"{out}/mixture.wav", m16, 16000); sf.write(f"{out}/vocals.wav", v16, 16000)
                sf.write(f"{ACC}/{track}.wav", m16 - v16, 16000)
                print(f"  [{k+1}/{len(names)}] {track}")
                break
            except Exception as e:
                wait = 10 * (attempt + 1)
                print(f"  [{k+1}/{len(names)}] {track}: download broke ({str(e)[:80]}); retry {attempt+1}/5 in {wait} s")
                time.sleep(wait)
                try: z = open_zip()
                except Exception: pass
        else:
            failed.append(track)
            subprocess.run(["rm", "-rf", out])
        subprocess.run(["rm", "-rf", tmp])
    got = len(glob.glob(f"{MUS}/*/*/vocals.wav"))
    print(f"  songs ready: {got}/{len(names)}" + (f"; skipped after retries: {failed}" if failed else ""))
    if got < 50:
        raise RuntimeError(f"only {got} MUSDB songs downloaded - Zenodo is struggling, try again later")
step("MUSDB18-HQ audio", get_musdb)

def get_musdb_lyrics():
    os.makedirs(f"{DATA}/musdb_lyrics", exist_ok=True)
    if len(glob.glob(f"{DATA}/musdb_lyrics/**/*.txt", recursive=True)) >= 100:
        print("lyrics files:", len(glob.glob(f"{DATA}/musdb_lyrics/**/*.txt", recursive=True))); return
    for part in ("train_lyrics.zip", "test_lyrics.zip"):
        z = f"{DATA}/{part}"
        if not os.path.exists(z):
            subprocess.run(["wget", "-q", "-O", z, f"https://zenodo.org/records/3989267/files/{part}?download=1"], check=True)
        zipfile.ZipFile(z).extractall(f"{DATA}/musdb_lyrics")
    print("lyrics files:", len(glob.glob(f"{DATA}/musdb_lyrics/**/*.txt", recursive=True)))
step("MUSDB18 lyrics extension", get_musdb_lyrics)

# 2. GTSinger English (8.7 GB)
def get_gtsinger():
    # GTSinger is ~20k small files and Hugging Face allows 5000 file requests per 5 minutes, so
    # fetch only what training reads (audio + word labels), and on a rate limit wait it out and
    # carry on: files already downloaded are skipped on the next attempt.
    import time
    from huggingface_hub import snapshot_download
    for attempt in range(15):
        try:
            snapshot_download("GTSinger/GTSinger", repo_type="dataset",
                              allow_patterns=["English/**/*.wav", "English/**/*.json"],
                              local_dir=f"{DATA}/GTSinger", max_workers=4,
                              token=os.environ.get("HF_TOKEN") or None)
            break
        except Exception as e:
            if "429" not in str(e):
                raise
            print(f"  Hugging Face rate limit (attempt {attempt+1}); waiting 5 min, then resuming...")
            time.sleep(310)
    else:
        print("  still rate-limited; continuing with the GTSinger files downloaded so far")
    print("wavs:", len(glob.glob(f"{DATA}/GTSinger/English/**/*.wav", recursive=True)))
if USE_GTSINGER:
    step("GTSinger English", get_gtsinger)
else:
    print("== GTSinger English: skipped (USE_GTSINGER = False; run 1 spent 90 min on rate limits and got 0 usable clips)")

# 3. Children's Song Dataset (1.9 GB)
def get_csd():
    if not glob.glob(f"{DATA}/CSD/**/english", recursive=True):
        subprocess.run(["wget", "-q", "-O", f"{DATA}/CSD.zip", "https://zenodo.org/records/4785016/files/CSD.zip?download=1"], check=True)
        zipfile.ZipFile(f"{DATA}/CSD.zip").extractall(f"{DATA}/CSD"); os.remove(f"{DATA}/CSD.zip")
step("Children's Song Dataset", get_csd)

# 4. Fallback accompaniment if MUSDB18-HQ didn't work: the free 7 s previews
def get_previews():
    os.makedirs(ACC, exist_ok=True)
    if len(os.listdir(ACC)) < 20:
        import musdb
        for t in musdb.DB(root=f"{DATA}/musdb7", download=True):
            sf.write(f"{ACC}/{t.name.replace('/', '_')}_preview.wav", t.targets["accompaniment"].audio, t.rate)
    print("accompaniment files:", len(os.listdir(ACC)))
step("accompaniment", get_previews)

# 5. JamendoLyrics (evaluation only)
def get_jamendo():
    if not os.path.isdir(f"{DATA}/jamendolyrics"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/f90/jamendolyrics", f"{DATA}/jamendolyrics"], check=True)
step("JamendoLyrics", get_jamendo)

import json
print(json.dumps(STATUS, indent=1))
print(subprocess.run(f"du -sh {DATA}/* ; df -h {DATA}", shell=True, capture_output=True, text=True).stdout)

## Training lists

In [ ]:
import glob, os, subprocess
TRAIN, VAL = [], []
def build(name, args):
    out = f"{DATA}/manifests/{name}"
    r = subprocess.run(f"python prepare_data.py {args} --out {out}", shell=True, text=True, capture_output=True)
    print(f"--- {name}\n" + r.stdout[-1500:] + (r.stderr[-1500:] if r.returncode else ""))
    if r.returncode == 0 and os.path.exists(f"{out}/train.jsonl") and os.path.getsize(f"{out}/train.jsonl") > 0:
        TRAIN.append(f"{out}/train.jsonl"); VAL.append(f"{out}/val.jsonl")

if glob.glob(f"{DATA}/musdb16k/*/*/mixture.wav") and glob.glob(f"{DATA}/musdb_lyrics/**/*.txt", recursive=True):
    build("musdb", f"musdb --musdb-root {DATA}/musdb16k --lyrics-dir {DATA}/musdb_lyrics")
if glob.glob(f"{DATA}/GTSinger/English/**/*.wav", recursive=True):
    build("gtsinger", f"utterances --root {DATA}/GTSinger/English")
csd = sorted(glob.glob(f"{DATA}/CSD/**/english", recursive=True))
if csd:
    lyr = f"{csd[0]}/lyric" if os.path.isdir(f"{csd[0]}/lyric") else f"{csd[0]}/txt"
    build("csd", f"longform --audio-dir {csd[0]}/wav --lyrics-dir {lyr}")
for i, d in enumerate(sorted({os.path.dirname(p) for p in glob.glob(f"{MY_MUSIC}/**/*.lrc", recursive=True)})):
    build(f"mylib{i}", f'lrc --music-dir "{d}"')
if not TRAIN:
    raise SystemExit("No training data could be prepared - see the messages above.")
if not any(("/musdb/" in t) or ("/mylib" in t) for t in TRAIN):
    # Children's songs alone (2 h, one singer style) aren't worth the GPU hours.
    raise SystemExit("No real-song data (MUSDB18 download failed and no songs of your own attached). "
                     "Stopping so you don't waste GPU time - run it again in a while.")
print("training on:", TRAIN)

## Baseline: the app's current model on real songs

In [ ]:
!python evaluate_alignment.py --jamendo {DATA}/jamendolyrics --models facebook/wav2vec2-base-960h --out {RUN}/baseline.json

## Train

In [ ]:
train_args, val_args = " ".join(TRAIN), " ".join(VAL)
!python train.py --base "{BASE}" --train {train_args} --val {val_args} \
  --accomp-dir {DATA}/accompaniment --out {RUN} --steps {STEPS} --lr {LR} --workers 2 --resume

## Export + compare

In [ ]:
import os
if not os.path.isdir(f"{RUN}/best"):
    raise SystemExit("Training produced no model (see the training cell above). Nothing to export.")
!python export_onnx.py --checkpoint {RUN}/best --out {RUN}/model_int8.onnx --version {VERSION}
# Fair comparison: the starting model squeezed to int8 the same way, so the only difference
# between the last two rows is the training (int8 rounding alone moved some songs in run 1).
!python export_onnx.py --checkpoint "{BASE}" --out {DATA}/start_int8.onnx --version start
models = f"{DATA}/start_int8.onnx {RUN}/model_int8.onnx"
!python evaluate_alignment.py --jamendo {DATA}/jamendolyrics --models {models} --out {RUN}/alignment_report.json

**Use the new model only if `model_int8.onnx` beats the others** on median error and "within 300 ms". Then send Claude the constants printed by the export step.

In [ ]:
import os
if HF_REPO and os.environ.get("HF_TOKEN"):
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ["HF_TOKEN"]); api.create_repo(HF_REPO, exist_ok=True)
    api.upload_file(path_or_fileobj=f"{RUN}/model_int8.onnx", path_in_repo=f"{VERSION}.onnx", repo_id=HF_REPO)
    print(f"MODEL_URL = https://huggingface.co/{HF_REPO}/resolve/main/{VERSION}.onnx")
else:
    print(f"Not uploaded. model_int8.onnx is in {RUN}")
if os.path.exists(f"{RUN}/model_int8.onnx") and os.path.exists(f"{RUN}/resume.pt"):
    os.remove(f"{RUN}/resume.pt")   # ~1 GB; only needed while training